In [ ]:
import os
import numpy as np
import pandas as pd

IN_DIR = "./pos_pre"
OUT_CSV = "./ensemble_probs_pos.csv"
REPS = list(range(1, 11))
REP_THRESH = {1: 0.44, 2: 0.66, 3: 0.55, 4: 0.76, 5: 0.60, 6: 0.32, 7: 0.53, 8: 0.57, 9: 0.46, 10: 0.50}
base = None
base_path = None
for rep in REPS:
    pth = os.path.join(IN_DIR, f"rep{rep}_test_probs.csv")
    if os.path.exists(pth):
        base = pd.read_csv(pth)
        base_path = pth
        break

if base is None:
    raise FileNotFoundError(f"No rep*_test_probs.csv found in {IN_DIR}")

required_cols = {"sequence", "true_len", "prob"}
if not required_cols.issubset(base.columns):
    raise ValueError(f"{base_path} must contain columns {required_cols}")

seq = base["sequence"].astype(str).tolist()
true_len = base["true_len"].to_numpy()

prob_dict = {}
label_dict = {}
missing = []

for rep in REPS:
    pth = os.path.join(IN_DIR, f"rep{rep}_test_probs.csv")
    if not os.path.exists(pth):
        missing.append(rep)
        continue

    df = pd.read_csv(pth)

    if df.shape[0] != len(seq) or not (df["sequence"].astype(str).tolist() == seq):
        raise ValueError(
            f"Sequence order mismatch in {pth}. "
            f"Ensure all reps scored the same TEST_CSV in same order."
        )

    prob = df["prob"].to_numpy(dtype=np.float32)
    prob_dict[rep] = prob
    label_dict[rep] = (prob > float(REP_THRESH[rep])).astype(np.int8)

if missing:
    print("WARNING: missing prob files for reps:", missing)

available_reps = [r for r in REPS if r in prob_dict]
if len(available_reps) == 0:
    raise ValueError("No available rep prob files found.")

prob_mat = np.stack([prob_dict[r] for r in available_reps], axis=1)
label_mat = np.stack([label_dict[r] for r in available_reps], axis=1)

out = pd.DataFrame({
    "sequence": seq,
    "true_len": true_len,
    "prob_mean": prob_mat.mean(axis=1),
    "prob_median": np.median(prob_mat, axis=1),
    "prob_std": prob_mat.std(axis=1),
    "label_sum": label_mat.sum(axis=1).astype(np.int32),
})

out = out.sort_values(["label_sum", "prob_mean"], ascending=[False, False]).reset_index(drop=True)
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)


,sequence,true_len,prob_mean,prob_median,prob_std,label_sum
0,CGCGAAGCGTTCAGGGTCTA,20,0.962061,0.969727,0.020227,10
1,CGCGACGGTTCGAACCGTCT,20,0.960254,0.972656,0.036717,10
2,CGCGACGATCCGCACAGCAC,20,0.959521,0.974121,0.044858,10
3,GCGATTCGGTCGCAGCGTA,19,0.958545,0.963623,0.026493,10
4,CGCGACGCGGCACAGTGCAA,20,0.958350,0.965088,0.021318,10
...,...,...,...,...,...,...
1199995,AGTCGGGATCGGGGGGGGGA,20,0.257257,0.226501,0.093686,0
1199996,TCGGTGGACGTCAAAATTGG,20,0.242639,0.166992,0.142044,0
1199997,TCGGTGGACGTAGGGGGGGA,20,0.238904,0.233704,0.118561,0
1199998,ACGAGGGCGTACCCCCCCCC,20,0.237134,0.245056,0.096074,0


In [ ]:
import pandas as pd
import os
OUT_DIR = "./test_ig_by_rep_pos/"
all_data = []
for rep in range(1, 11):

    file_path = os.path.join(OUT_DIR, f"rep{rep}_test_top_by_final.csv")
    

    if os.path.exists(file_path):

        df = pd.read_csv(file_path)
        all_data.append(df)
    else:
        print(f"File for Rep {rep} does not exist.")
combined_df = pd.concat(all_data, ignore_index=True)
average_values = combined_df.groupby("sequence")[["prob", "importance"]].mean().reset_index()
average_values.to_csv(os.path.join(OUT_DIR, "average_prob_importance_per_sequence.csv"), index=False)
print("All done. Average values per sequence saved to:", os.path.abspath(os.path.join(OUT_DIR, "average_prob_importance_per_sequence.csv")))